---
## Step 1 — Install Dependencies
> Run this cell once, then **restart the runtime** (Runtime → Restart runtime).

In [ ]:
# ── Core ML ───────────────────────────────────────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# ── HuggingFace stack ─────────────────────────────────────────────────────
!pip install -q transformers==4.40.0 accelerate peft bitsandbytes datasets

# ── Vision & pose ─────────────────────────────────────────────────────────
!pip install -q ultralytics opencv-python-headless pillow albumentations

# ── Vector search ─────────────────────────────────────────────────────────
!pip install -q faiss-gpu

# ── Utilities ─────────────────────────────────────────────────────────────
!pip install -q pandas numpy scipy scikit-learn requests tqdm wandb matplotlib seaborn

print('\n✓ All packages installed. NOW RESTART THE RUNTIME before continuing.')

### Verify installation (run after restart)

In [ ]:
import torch, transformers, peft, accelerate, ultralytics, faiss, wandb
import pandas, numpy, sklearn

print(f'PyTorch:       {torch.__version__}')
print(f'Transformers:  {transformers.__version__}')
print(f'PEFT:          {peft.__version__}')
print(f'Ultralytics:   {ultralytics.__version__}')
print(f'FAISS:         {faiss.__version__}')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nGPU:  {name}')
    print(f'VRAM: {vram:.1f} GB')
    assert vram >= 14, 'Need at least 14 GB VRAM — switch to T4 in runtime settings'
    print('✓ T4 detected — ready to proceed')
else:
    print('⚠ No GPU detected — switch to T4 GPU in runtime settings')

---
## Step 2 — Configuration
All hyperparameters in one place. Adjust here before running downstream cells.

In [ ]:
import os, random
import numpy as np
import torch

# ── Paths ─────────────────────────────────────────────────────────────────
ROOT_DIR   = '/content/painting_similarity'
DATA_DIR   = f'{ROOT_DIR}/data'
IMG_DIR    = f'{DATA_DIR}/images'
EMBED_DIR  = f'{ROOT_DIR}/embeddings'
CKPT_DIR   = f'{ROOT_DIR}/checkpoints'
INDEX_DIR  = f'{ROOT_DIR}/index'
RESULTS_DIR= f'{ROOT_DIR}/results'

MASTER_CSV    = f'{DATA_DIR}/master_dataset.csv'
SIGLIP_EMBED  = f'{EMBED_DIR}/siglip_embeddings.npy'
DINO_EMBED    = f'{EMBED_DIR}/dino_embeddings.npy'
POSE_EMBED    = f'{EMBED_DIR}/pose_embeddings.npy'
FUSED_EMBED   = f'{EMBED_DIR}/fused_embeddings.npy'
EMBED_IDS     = f'{EMBED_DIR}/object_ids.npy'
FAISS_INDEX   = f'{INDEX_DIR}/painting_index.faiss'
FAISS_META    = f'{INDEX_DIR}/index_metadata.pkl'

for d in [DATA_DIR,IMG_DIR,EMBED_DIR,CKPT_DIR,INDEX_DIR,RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hardware ──────────────────────────────────────────────────────────────
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_FP16 = True
SEED     = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ── Models ────────────────────────────────────────────────────────────────
SIGLIP_MODEL    = 'google/siglip-large-patch16-384'
DINO_MODEL      = 'facebook/dinov2-large'
YOLO_POSE_MODEL = 'yolov8l-pose.pt'

# ── Preprocessing ─────────────────────────────────────────────────────────
SIGLIP_IMG_SIZE = 384
DINO_IMG_SIZE   = 518
MIN_IMG_DIM     = 64

# ── Data ──────────────────────────────────────────────────────────────────
MAX_IMAGES       = 50_000   # Set to None for full 130K dataset
DOWNLOAD_WORKERS = 8
DOWNLOAD_TIMEOUT = 15

# ── Embeddings ────────────────────────────────────────────────────────────
EMBED_BATCH_SIZE = 32
EMBED_DIM_SIGLIP = 1024
EMBED_DIM_DINO   = 1024
POSE_KEYPOINTS   = 17
POSE_EMBED_DIM   = 34

# ── LoRA fine-tuning ──────────────────────────────────────────────────────
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'out_proj']
TRAIN_BATCH_SIZE    = 32
GRAD_ACCUM_STEPS    = 4
LEARNING_RATE       = 2e-4
WEIGHT_DECAY        = 0.01
WARMUP_STEPS        = 200
MAX_TRAIN_STEPS     = 3000
SAVE_EVERY_N_STEPS  = 500
TRIPLET_MARGIN      = 0.3

# ── Fusion ────────────────────────────────────────────────────────────────
FUSION_HIDDEN_DIM = 512
FUSION_OUTPUT_DIM = 512
FUSION_N_HEADS    = 8
FUSION_DROPOUT    = 0.1
FUSION_BATCH_SIZE = 64

# ── FAISS ─────────────────────────────────────────────────────────────────
FAISS_NLIST  = 256
FAISS_M_PQ   = 64
FAISS_NBITS  = 8
FAISS_NPROBE = 32

# ── Eval ──────────────────────────────────────────────────────────────────
TOP_K         = 20
EVAL_K_VALUES = [1, 5, 10, 20]
EVAL_N_QUERIES= 500

# ── W&B (optional) ────────────────────────────────────────────────────────
WANDB_PROJECT = 'nga-painting-similarity'

print(f'Device: {DEVICE}  |  fp16: {USE_FP16}')
print('✓ Configuration loaded')

---
## Step 3 — NGA Dataset Collection
Downloads `objects.csv` and `published_images.csv` from the NGA GitHub repo,
joins them, filters to paintings, and downloads the actual image files in parallel.

⏱ **~30–45 min** depending on network speed.

In [ ]:
import requests, time
import pandas as pd
from io import BytesIO
from pathlib import Path
from PIL import Image, ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

NGA_BASE = 'https://raw.githubusercontent.com/NationalGalleryOfArt/opendata/main/data'

def download_csv(filename, url):
    dest = Path(DATA_DIR) / filename
    if dest.exists():
        print(f'  [cached] {filename}')
        return pd.read_csv(dest, low_memory=False)
    print(f'  Downloading {filename}...')
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f'  Saved {filename} ({len(r.content)/1e6:.1f} MB)')
    return pd.read_csv(BytesIO(r.content), low_memory=False)

objects_df = download_csv('objects.csv',          f'{NGA_BASE}/objects.csv')
images_df  = download_csv('published_images.csv', f'{NGA_BASE}/published_images.csv')

objects_df.columns = [c.strip().lower() for c in objects_df.columns]
images_df.columns  = [c.strip().lower() for c in images_df.columns]

print(f'\nobjects.csv rows:          {len(objects_df):,}')
print(f'published_images.csv rows: {len(images_df):,}')
print(f'objects columns: {list(objects_df.columns)}')

In [ ]:
# ── Join and filter ───────────────────────────────────────────────────────
img_url_col = next((c for c in ['iiifthumburl','iiifurl','thumbnailurl','url']
                    if c in images_df.columns), None)
assert img_url_col, f'No image URL column found. Available: {list(images_df.columns)}'
print(f'Image URL column: {img_url_col}')

obj_cols = [c for c in ['objectid','title','attribution','displaydate',
            'classification','medium','subjectterms','nationality'] if c in objects_df.columns]
objects_clean = objects_df[obj_cols].copy()

images_clean = images_df[['objectid', img_url_col]].copy()
images_clean = images_clean.rename(columns={img_url_col: 'image_url'})
images_clean = images_clean.groupby('objectid', as_index=False).first()

master = objects_clean.merge(images_clean, on='objectid', how='inner')
print(f'After join: {len(master):,} paintings with images')

# Filter to paintings only
if 'classification' in master.columns:
    master = master[master['classification'].str.lower().str.contains('painting', na=False)]
    print(f'After painting filter: {len(master):,}')

master = master.dropna(subset=['image_url'])
master = master[master['image_url'].str.startswith('http')]
master = master.reset_index(drop=True)

if MAX_IMAGES and len(master) > MAX_IMAGES:
    master = master.sample(n=MAX_IMAGES, random_state=SEED).reset_index(drop=True)
    print(f'Dev cap applied: {MAX_IMAGES:,} paintings')

print(f'\n✓ Final dataset: {len(master):,} paintings')
master.head(3)

In [ ]:
# ── Build download URLs and download images ───────────────────────────────
def build_url(raw_url, size=512):
    if not isinstance(raw_url, str): return raw_url
    base = raw_url.split('/full/')[0] if '/full/' in raw_url else raw_url.rstrip('/')
    return f'{base}/full/!{size},{size}/0/default.jpg'

master['download_url'] = master['image_url'].apply(lambda u: build_url(u, 512))
master['local_path']   = master['objectid'].apply(
    lambda oid: str(Path(IMG_DIR) / f'{oid}.jpg'))

def download_one(row):
    oid, url, path = row['objectid'], row['download_url'], row['local_path']
    if Path(path).exists(): return (oid, 'cached')
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=DOWNLOAD_TIMEOUT, stream=True)
            if r.status_code == 404: return (oid, 'not_found')
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert('RGB')
            if min(img.size) < MIN_IMG_DIM: return (oid, 'too_small')
            img.save(path, 'JPEG', quality=90)
            return (oid, 'ok')
        except Exception as e:
            if attempt == 2: return (oid, f'err:{str(e)[:30]}')
            time.sleep(0.5)

rows = master.to_dict('records')
results = []
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
    futures = {ex.submit(download_one, r): r for r in rows}
    for f in tqdm(as_completed(futures), total=len(futures), desc='Downloading images'):
        results.append(f.result())

status = pd.DataFrame(results, columns=['objectid','status'])
print(status['status'].value_counts().to_string())

ok_ids = set(status[status['status'].isin(['ok','cached'])]['objectid'])
master_clean = master[master['objectid'].isin(ok_ids)].reset_index(drop=True)
master_clean.to_csv(MASTER_CSV, index=False)
print(f'\n✓ {len(master_clean):,} paintings ready  →  {MASTER_CSV}')

---
## Step 4 — Image Preprocessing & Dataset Class
Defines the transforms for both models and the `PaintingDataset` used throughout the pipeline.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from typing import Optional, Callable, Dict, List

SIGLIP_MEAN = [0.5, 0.5, 0.5]
SIGLIP_STD  = [0.5, 0.5, 0.5]
DINO_MEAN   = [0.485, 0.456, 0.406]
DINO_STD    = [0.229, 0.224, 0.225]

def get_siglip_transform(augment=False):
    base = [
        transforms.Resize((SIGLIP_IMG_SIZE, SIGLIP_IMG_SIZE),
                           interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(SIGLIP_MEAN, SIGLIP_STD),
    ]
    if augment:
        return transforms.Compose([
            transforms.Resize((SIGLIP_IMG_SIZE, SIGLIP_IMG_SIZE),
                               interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.1)], p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(SIGLIP_MEAN, SIGLIP_STD),
        ])
    return transforms.Compose(base)

def get_dino_transform(augment=False):
    return transforms.Compose([
        transforms.Resize((DINO_IMG_SIZE, DINO_IMG_SIZE),
                           interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(DINO_MEAN, DINO_STD),
    ])

class PaintingDataset(Dataset):
    def __init__(self, df, transform=None, mode='embed', model='siglip'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.mode = mode
        self.model = model
        if mode == 'triplet': self._build_tag_index()

    def _build_tag_index(self):
        self.tag_to_indices = {}
        col = 'subjectterms' if 'subjectterms' in self.df.columns else 'classification'
        for idx, row in self.df.iterrows():
            for tag in str(row.get(col,'')).lower().split('|'):
                tag = tag.strip()
                if tag: self.tag_to_indices.setdefault(tag,[]).append(idx)
        self.tag_to_indices = {t:v for t,v in self.tag_to_indices.items() if len(v)>=2}
        self.valid_indices = list(set(i for v in self.tag_to_indices.values() for i in v))

    def _load(self, path):
        try: return Image.open(path).convert('RGB')
        except:
            sz = SIGLIP_IMG_SIZE if self.model=='siglip' else DINO_IMG_SIZE
            return Image.new('RGB',(sz,sz),(0,0,0))

    def __len__(self):
        return len(self.valid_indices) if self.mode=='triplet' else len(self.df)

    def __getitem__(self, idx):
        if self.mode == 'triplet': return self._triplet(idx)
        row = self.df.iloc[idx]
        img = self._load(str(row['local_path']))
        tensor = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return {'pixel_values': tensor, 'objectid': str(row['objectid']), 'idx': idx}

    def _triplet(self, idx):
        import random as rnd
        anchor_idx = self.valid_indices[idx]
        anchor_row = self.df.iloc[anchor_idx]
        col = 'subjectterms' if 'subjectterms' in self.df.columns else 'classification'
        anchor_tags = {t.strip().lower() for t in str(anchor_row.get(col,'')).split('|') if t.strip()}
        shared = [t for t in anchor_tags if t in self.tag_to_indices]
        if shared:
            pool = [i for i in self.tag_to_indices[rnd.choice(shared)] if i != anchor_idx]
            pos_idx = rnd.choice(pool) if pool else anchor_idx
        else: pos_idx = anchor_idx
        neg_pool = self.df[~self.df.index.isin([anchor_idx, pos_idx])].index.tolist()
        neg_idx = rnd.choice(neg_pool) if neg_pool else 0
        def lt(i):
            row = self.df.iloc[i]
            img = self._load(str(row['local_path']))
            return self.transform(img) if self.transform else transforms.ToTensor()(img)
        return {'anchor': lt(anchor_idx), 'positive': lt(pos_idx), 'negative': lt(neg_idx),
                'anchor_id': str(anchor_row['objectid']),
                'positive_id': str(self.df.iloc[pos_idx]['objectid']),
                'negative_id': str(self.df.iloc[neg_idx]['objectid'])}

# Validate images
df = pd.read_csv(MASTER_CSV, low_memory=False)
bad = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc='Validating'):
    p = str(row.get('local_path',''))
    if not Path(p).exists(): bad.append(idx); continue
    try:
        img = Image.open(p); img.verify()
        img = Image.open(p).convert('RGB')
        if min(img.size) < MIN_IMG_DIM: bad.append(idx)
    except: bad.append(idx)

df = df.drop(index=bad).reset_index(drop=True)
df.to_csv(MASTER_CSV, index=False)
print(f'✓ Validation complete: {len(df):,} clean images  ({len(bad)} removed)')

---
## Step 5 — Embedding Extraction: SigLIP + DINOv2
Runs both vision encoders over all paintings in fp16 and saves embedding arrays.

⏱ **~25 min** · VRAM peak ~6 GB

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoProcessor

@torch.no_grad()
def extract_embeddings(model, loader, stream, device):
    all_embs, all_ids = [], []
    for batch in tqdm(loader, desc=f'{stream} embeddings'):
        pv = batch['pixel_values'].to(device)
        if USE_FP16: pv = pv.half()
        if stream == 'siglip':
            out = model.vision_model(pixel_values=pv)
            emb = F.normalize(out.pooler_output.float(), dim=-1)
        else:  # dino
            out = model(pixel_values=pv)
            emb = F.normalize(out.last_hidden_state[:,0,:].float(), dim=-1)
        all_embs.append(emb.cpu().numpy())
        all_ids.extend(batch['objectid'])
    return np.vstack(all_embs), all_ids

# ── SigLIP ────────────────────────────────────────────────────────────────
if Path(SIGLIP_EMBED).exists():
    print('[cached] SigLIP embeddings found')
    sig_embs = np.load(SIGLIP_EMBED)
    sig_ids  = list(np.load(EMBED_IDS, allow_pickle=True))
else:
    print('Loading SigLIP...')
    sig_model = AutoModel.from_pretrained(SIGLIP_MODEL,
                    torch_dtype=torch.float16 if USE_FP16 else torch.float32).to(DEVICE).eval()
    sig_ds  = PaintingDataset(df, transform=get_siglip_transform(), mode='embed', model='siglip')
    sig_ldr = DataLoader(sig_ds, batch_size=EMBED_BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
    sig_embs, sig_ids = extract_embeddings(sig_model, sig_ldr, 'siglip', DEVICE)
    np.save(SIGLIP_EMBED, sig_embs)
    np.save(EMBED_IDS, np.array(sig_ids, dtype=object))
    del sig_model; torch.cuda.empty_cache()
    print(f'✓ SigLIP embeddings saved  {sig_embs.shape}')

# ── DINOv2 ────────────────────────────────────────────────────────────────
if Path(DINO_EMBED).exists():
    print('[cached] DINOv2 embeddings found')
    dino_embs = np.load(DINO_EMBED)
else:
    print('Loading DINOv2...')
    dino_model = AutoModel.from_pretrained(DINO_MODEL,
                    torch_dtype=torch.float16 if USE_FP16 else torch.float32).to(DEVICE).eval()
    dino_ds  = PaintingDataset(df, transform=get_dino_transform(), mode='embed', model='dino')
    dino_ldr = DataLoader(dino_ds, batch_size=EMBED_BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
    dino_embs, _ = extract_embeddings(dino_model, dino_ldr, 'dino', DEVICE)
    np.save(DINO_EMBED, dino_embs)
    del dino_model; torch.cuda.empty_cache()
    print(f'✓ DINOv2 embeddings saved  {dino_embs.shape}')

print(f'SigLIP: {sig_embs.shape}  |  DINOv2: {dino_embs.shape}')

In [ ]:
# ── Cosine similarity sanity check ────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

id_to_meta = {str(r['objectid']): r for _, r in df.iterrows()}
for _ in range(3):
    qi = random.randint(0, len(sig_ids)-1)
    qid = sig_ids[qi]
    sims = cos_sim(sig_embs[qi:qi+1], sig_embs)[0]
    top5 = np.argsort(-sims)[1:6]
    q = id_to_meta.get(qid, {})
    print(f"\nQuery: {str(q.get('title',''))[:50]}")
    print(f"Tags:  {str(q.get('subjectterms',''))[:60]}")
    for rank, ridx in enumerate(top5, 1):
        rid = sig_ids[ridx]
        r = id_to_meta.get(rid, {})
        print(f'  {rank}. [{sims[ridx]:.3f}] {str(r.get("title",""))[:45]}')

---
## Step 6 — Pose Extraction: YOLOv8-large-pose
Detects 17 COCO body keypoints per painting. Keypoints are normalised
to be position- and scale-invariant, producing a 34-dim pose vector.

⏱ **~15 min** · VRAM ~1.2 GB

In [ ]:
from ultralytics import YOLO

COCO_KPS = ['nose','l_eye','r_eye','l_ear','r_ear','l_shoulder','r_shoulder',
            'l_elbow','r_elbow','l_wrist','r_wrist','l_hip','r_hip',
            'l_knee','r_knee','l_ankle','r_ankle']

def extract_pose_vec(result, w, h, conf_thr=0.3):
    vec = np.zeros(POSE_EMBED_DIM, dtype=np.float32)
    if result.keypoints is None or len(result.keypoints)==0: return vec
    kdata = result.keypoints.data
    if kdata is None or kdata.shape[0]==0: return vec
    boxes = result.boxes
    if boxes is not None and len(boxes)>1:
        best = int((boxes.xywh[:,2]*boxes.xywh[:,3]).argmax().item())
    else: best=0
    kpts = kdata[best].cpu().numpy()
    for i in range(min(POSE_KEYPOINTS, kpts.shape[0])):
        x,y,c = kpts[i]
        if c>=conf_thr:
            vec[i*2]   = x/w
            vec[i*2+1] = y/h
    return vec

def norm_pose_vec(vec):
    xs, ys = vec[0::2], vec[1::2]
    ax, ay = xs[xs!=0], ys[ys!=0]
    if len(ax)==0: return vec
    cx, cy = ax.mean(), ay.mean()
    out = vec.copy()
    out[0::2] = np.where(xs!=0, xs-cx, 0.)
    out[1::2] = np.where(ys!=0, ys-cy, 0.)
    return out

if Path(POSE_EMBED).exists():
    print('[cached] Pose embeddings found')
    pose_embs = np.load(POSE_EMBED)
else:
    yolo = YOLO(YOLO_POSE_MODEL).to(DEVICE)
    paths   = df['local_path'].tolist()
    obj_ids = [str(x) for x in df['objectid'].tolist()]
    all_poses, n_det = [], 0
    for start in tqdm(range(0, len(paths), 16), desc='Pose extraction'):
        batch_p = paths[start:start+16]
        imgs    = []
        dims    = []
        for p in batch_p:
            try:
                img = Image.open(p).convert('RGB')
                imgs.append(img); dims.append((img.width,img.height))
            except:
                imgs.append(Image.new('RGB',(224,224))); dims.append((224,224))
        results = yolo.predict(source=imgs, conf=0.3, verbose=False,
                               device=DEVICE, imgsz=640)
        for res,(w,h) in zip(results, dims):
            v = norm_pose_vec(extract_pose_vec(res,w,h))
            all_poses.append(v)
            if v.sum()!=0: n_det+=1
    pose_embs = np.vstack(all_poses).astype(np.float32)
    np.save(POSE_EMBED, pose_embs)
    del yolo; torch.cuda.empty_cache()
    print(f'✓ Pose saved {pose_embs.shape}  |  figures detected: {n_det:,}/{len(paths):,}')

nz = (~(pose_embs==0).all(axis=1)).sum()
print(f'Pose vectors with detected figure: {nz:,} / {len(pose_embs):,} ({100*nz/len(pose_embs):.1f}%)')

---
## Step 7 — LoRA Fine-Tuning (SigLIP + Triplet Loss)
Adapts SigLIP to the art domain. Only ~2M of 307M parameters are trained.
Training automatically resumes after a Colab session disconnect.

⏱ **~45 min** · VRAM peak ~6 GB

In [ ]:
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from transformers import AutoModel, get_cosine_schedule_with_warmup
from torch.utils.data import DataLoader, random_split

class TripletLoss(nn.Module):
    def __init__(self, margin=TRIPLET_MARGIN):
        super().__init__()
        self.m = margin
    def forward(self, a, p, n):
        a,p,n = F.normalize(a,dim=-1), F.normalize(p,dim=-1), F.normalize(n,dim=-1)
        d_ap = (a-p).pow(2).sum(-1).clamp(1e-8).sqrt()
        d_an = (a-n).pow(2).sum(-1).clamp(1e-8).sqrt()
        return F.relu(d_ap - d_an + self.m).mean()

class SigLIPEmbedder(nn.Module):
    def __init__(self, vm): super().__init__(); self.vm = vm
    def forward(self, pv):
        return F.normalize(self.vm(pixel_values=pv).pooler_output.float(), dim=-1)

def build_lora_model():
    base = AutoModel.from_pretrained(SIGLIP_MODEL,
               torch_dtype=torch.float16 if USE_FP16 else torch.float32)
    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias='none')
    vm = get_peft_model(base.vision_model, lora_cfg)
    vm.print_trainable_parameters()
    return SigLIPEmbedder(vm).to(DEVICE)

final_adapter = Path(CKPT_DIR)/'final_lora_adapter'
if final_adapter.exists():
    print('[cached] Fine-tuned adapter found — skipping training')
else:
    embedder  = build_lora_model()
    criterion = TripletLoss()
    aug_tf = get_siglip_transform(augment=True)
    full_ds  = PaintingDataset(df, transform=aug_tf, mode='triplet', model='siglip')
    val_n    = max(100, int(0.1*len(full_ds)))
    trn_ds, val_ds = random_split(full_ds,[len(full_ds)-val_n,val_n],
                          generator=torch.Generator().manual_seed(SEED))
    trn_ldr = DataLoader(trn_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)
    val_ldr = DataLoader(val_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
    opt     = torch.optim.AdamW(
        [p for p in embedder.parameters() if p.requires_grad],
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sch = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, MAX_TRAIN_STEPS)
    scaler  = torch.cuda.amp.GradScaler(enabled=USE_FP16)
    step, best_val = 0, float('inf')
    # Resume from last checkpoint
    ckpts = sorted(Path(CKPT_DIR).glob('step_*'))
    if ckpts:
        ck = torch.load(str(ckpts[-1])/'model.pt', map_location=DEVICE)
        embedder.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])
        step = ck['step']
        print(f'Resumed from step {step}')
    embedder.train()
    run_loss = 0.0
    print(f'Training from step {step}/{MAX_TRAIN_STEPS}...')
    for epoch in range(9999):
        for batch in trn_ldr:
            if step >= MAX_TRAIN_STEPS: break
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                a = embedder(batch['anchor'].to(DEVICE))
                p = embedder(batch['positive'].to(DEVICE))
                n = embedder(batch['negative'].to(DEVICE))
                loss = criterion(a,p,n) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step+1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(embedder.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sch.step()
            run_loss += loss.item()*GRAD_ACCUM_STEPS
            step += 1
            if step % 100 == 0:
                print(f'  step {step:4d} | loss={run_loss/100:.4f} | lr={sch.get_last_lr()[0]:.2e}')
                run_loss = 0.0
            if step % SAVE_EVERY_N_STEPS == 0:
                ck_path = Path(CKPT_DIR)/f'step_{step:05d}'
                ck_path.mkdir(exist_ok=True)
                torch.save({'model':embedder.state_dict(),'opt':opt.state_dict(),'step':step},
                           str(ck_path/'model.pt'))
        if step >= MAX_TRAIN_STEPS: break
    embedder.vm.save_pretrained(str(final_adapter))
    print(f'✓ Fine-tuning complete  →  {final_adapter}')
    del embedder; torch.cuda.empty_cache()

In [ ]:
# ── Re-extract SigLIP embeddings with fine-tuned weights ──────────────────
from peft import PeftModel

if Path(SIGLIP_EMBED.replace('.npy','_finetuned.npy')).exists():
    print('[cached] Fine-tuned SigLIP embeddings exist')
else:
    print('Re-extracting with fine-tuned SigLIP...')
    base = AutoModel.from_pretrained(SIGLIP_MODEL, torch_dtype=torch.float16)
    ft_model = PeftModel.from_pretrained(base.vision_model, str(final_adapter))
    ft_model = ft_model.to(DEVICE).eval()
    sig_ds = PaintingDataset(df, transform=get_siglip_transform(), mode='embed', model='siglip')
    sig_ldr= DataLoader(sig_ds, batch_size=EMBED_BATCH_SIZE, shuffle=False, num_workers=2)
    all_embs = []
    with torch.no_grad():
        for batch in tqdm(sig_ldr, desc='Re-extracting SigLIP'):
            pv = batch['pixel_values'].to(DEVICE).half()
            out = ft_model(pixel_values=pv)
            emb = F.normalize(out.pooler_output.float(), dim=-1)
            all_embs.append(emb.cpu().numpy())
    sig_embs = np.vstack(all_embs)
    np.save(SIGLIP_EMBED, sig_embs)
    print(f'✓ Fine-tuned SigLIP embeddings saved  {sig_embs.shape}')
    del ft_model; torch.cuda.empty_cache()

---
## Step 8 — Cross-Attention Fusion Head
Combines SigLIP (semantic) + DINOv2 (structural) + Pose (body keypoints)
into a single 512-dim vector via multi-head cross-attention.

⏱ **~10 min**

In [ ]:
class CrossAttentionFusionHead(nn.Module):
    def __init__(self, siglip_dim=EMBED_DIM_SIGLIP, dino_dim=EMBED_DIM_DINO,
                 pose_dim=POSE_EMBED_DIM, d_model=FUSION_HIDDEN_DIM,
                 n_heads=FUSION_N_HEADS, output_dim=FUSION_OUTPUT_DIM, dropout=FUSION_DROPOUT):
        super().__init__()
        self.proj_sig  = nn.Sequential(nn.Linear(siglip_dim, d_model), nn.LayerNorm(d_model))
        self.proj_dino = nn.Sequential(nn.Linear(dino_dim, d_model),   nn.LayerNorm(d_model))
        self.proj_pose = nn.Sequential(
            nn.Linear(pose_dim,128), nn.ReLU(), nn.Linear(128, d_model), nn.LayerNorm(d_model))
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer    = nn.TransformerEncoder(enc, num_layers=2)
        self.stream_weights = nn.Parameter(torch.ones(3)/3)
        self.out_proj       = nn.Sequential(nn.Linear(d_model, output_dim), nn.LayerNorm(output_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, sig, dino, pose):
        s = self.proj_sig(sig).unsqueeze(1)
        d = self.proj_dino(dino).unsqueeze(1)
        p = self.proj_pose(pose).unsqueeze(1)
        seq = torch.cat([s,d,p], dim=1)
        out = self.transformer(seq)
        w   = F.softmax(self.stream_weights, dim=0)
        fused = (out * w.view(1,3,1)).sum(dim=1)
        return F.normalize(self.out_proj(fused), dim=-1)

head_path = Path(CKPT_DIR)/'fusion_head.pt'
if Path(FUSED_EMBED).exists():
    print('[cached] Fused embeddings found')
    fused_embs = np.load(FUSED_EMBED)
else:
    # Build triplet supervision from tag labels
    tag_to_idx = {}
    col = 'subjectterms' if 'subjectterms' in df.columns else 'classification'
    for idx, row in df.iterrows():
        for t in str(row.get(col,'')).lower().split('|'):
            t = t.strip()
            if t: tag_to_idx.setdefault(t,[]).append(idx)
    tag_to_idx = {t:v for t,v in tag_to_idx.items() if len(v)>=2}
    valid_tags = list(tag_to_idx.keys())
    triplets = []
    for _ in range(100_000):
        t  = random.choice(valid_tags)
        a,p= random.sample(tag_to_idx[t],2)
        nt = random.choice([x for x in valid_tags if x!=t])
        n  = random.choice(tag_to_idx[nt])
        triplets.append((a,p,n))
    # Train fusion head
    fusion_model = CrossAttentionFusionHead().to(DEVICE)
    crit = nn.TripletMarginLoss(margin=TRIPLET_MARGIN, p=2)
    opt  = torch.optim.AdamW(fusion_model.parameters(), lr=1e-3, weight_decay=1e-4)
    T_sig  = torch.from_numpy(np.load(SIGLIP_EMBED)).float()
    T_dino = torch.from_numpy(np.load(DINO_EMBED)).float()
    T_pose = torch.from_numpy(np.load(POSE_EMBED)).float()
    BS = FUSION_BATCH_SIZE
    for epoch in range(10):
        random.shuffle(triplets)
        total, nb = 0., 0
        for s in tqdm(range(0,len(triplets),BS), desc=f'Fusion epoch {epoch+1}/10', leave=False):
            b = triplets[s:s+BS]
            ai,pi,ni = map(list,zip(*b))
            a = fusion_model(T_sig[ai].to(DEVICE), T_dino[ai].to(DEVICE), T_pose[ai].to(DEVICE))
            p = fusion_model(T_sig[pi].to(DEVICE), T_dino[pi].to(DEVICE), T_pose[pi].to(DEVICE))
            n = fusion_model(T_sig[ni].to(DEVICE), T_dino[ni].to(DEVICE), T_pose[ni].to(DEVICE))
            loss = crit(a,p,n)
            opt.zero_grad(); loss.backward(); 
            torch.nn.utils.clip_grad_norm_(fusion_model.parameters(),1.0); opt.step()
            total+=loss.item(); nb+=1
        print(f'  Epoch {epoch+1}: loss={total/nb:.4f}')
    torch.save(fusion_model.state_dict(), str(head_path))
    # Generate all fused embeddings
    fusion_model.eval()
    all_fused = []
    with torch.no_grad():
        for s in tqdm(range(0,len(T_sig),512), desc='Fusing embeddings'):
            f = fusion_model(T_sig[s:s+512].to(DEVICE),
                             T_dino[s:s+512].to(DEVICE),
                             T_pose[s:s+512].to(DEVICE))
            all_fused.append(f.cpu().numpy())
    fused_embs = np.vstack(all_fused).astype(np.float32)
    np.save(FUSED_EMBED, fused_embs)
    print(f'✓ Fused embeddings saved  {fused_embs.shape}')

print(f'Fused shape: {fused_embs.shape}  |  norm range: [{np.linalg.norm(fused_embs,axis=1).min():.4f}, {np.linalg.norm(fused_embs,axis=1).max():.4f}]')

---
## Step 9 — FAISS IVF-PQ Index
Builds a GPU-accelerated approximate nearest-neighbour index.
For 50K paintings: **~0.5 ms per query** after indexing.

⏱ **~3 min**

In [ ]:
import faiss, pickle

object_ids = list(np.load(EMBED_IDS, allow_pickle=True).astype(str))
id_to_row  = {oid:i for i,oid in enumerate(object_ids)}
row_to_id  = {i:oid for i,oid in enumerate(object_ids)}

meta_dict = {
    str(r['objectid']): {
        'title':       str(r.get('title','')),
        'attribution': str(r.get('attribution','')),
        'date':        str(r.get('displaydate','')),
        'tags':        str(r.get('subjectterms','')),
        'local_path':  str(r.get('local_path','')),
        'image_url':   str(r.get('image_url','')),
    } for _, r in df.iterrows()}

metadata = {'object_ids':object_ids,'id_to_row':id_to_row,
            'row_to_id':row_to_id,'painting_meta':meta_dict}

if Path(FAISS_INDEX).exists():
    print('[cached] FAISS index found')
    index = faiss.read_index(FAISS_INDEX)
    index.nprobe = FAISS_NPROBE
else:
    embs = fused_embs.astype('float32')
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs  = embs / (norms + 1e-8)
    N, D = embs.shape
    quantiser = faiss.IndexFlatIP(D)
    index = faiss.IndexIVFPQ(quantiser,D,FAISS_NLIST,FAISS_M_PQ,FAISS_NBITS,
                              faiss.METRIC_INNER_PRODUCT)
    try:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res,0,index)
        print('Using GPU FAISS')
        use_gpu = True
    except: use_gpu = False
    print(f'Training IVF index on {N:,} vectors...')
    index.train(embs)
    for s in tqdm(range(0,N,10000), desc='Adding to FAISS'):
        index.add(embs[s:s+10000])
    if use_gpu: index = faiss.index_gpu_to_cpu(index)
    faiss.write_index(index, FAISS_INDEX)
    with open(FAISS_META,'wb') as f: pickle.dump(metadata,f)
    index.nprobe = FAISS_NPROBE
    print(f'✓ Index built  {index.ntotal:,} vectors  →  {FAISS_INDEX}')

print(f'Index: {index.ntotal:,} paintings ready')

---
## Step 10 — Query the Similarity System
Find similar paintings by NGA object ID or uploaded image file.
Try all three modes: `semantic`, `structural`, `fused`.

In [ ]:
# ── Load embedding matrices for all modes ─────────────────────────────────
mode_embeddings = {
    'semantic':   np.load(SIGLIP_EMBED).astype('float32'),
    'structural': np.load(DINO_EMBED).astype('float32'),
    'fused':      np.load(FUSED_EMBED).astype('float32'),
}
# Normalise all
for k,v in mode_embeddings.items():
    n = np.linalg.norm(v, axis=1, keepdims=True)
    mode_embeddings[k] = v / (n+1e-8)

def search(query_oid, mode='fused', top_k=TOP_K):
    row = id_to_row.get(str(query_oid))
    if row is None: print(f'Object ID {query_oid} not in index'); return []
    embs  = mode_embeddings[mode]
    q_emb = embs[row:row+1].astype('float32')
    D,I   = index.search(q_emb, top_k+5)
    results = []
    for dist,ridx in zip(D[0],I[0]):
        if ridx<0: continue
        rid = row_to_id.get(int(ridx))
        if not rid or str(rid)==str(query_oid): continue
        m = meta_dict.get(str(rid),{})
        results.append({'id':rid,'sim':float(dist),
                        'title':m.get('title',''),'tags':m.get('tags',''),'path':m.get('local_path','')})
        if len(results)>=top_k: break
    return results

def display(query_oid, results):
    q = meta_dict.get(str(query_oid),{})
    print(f"\n{'─'*70}")
    print(f"  Query: {q.get('title','')[:60]}")
    print(f"  Tags:  {q.get('tags','')[:65]}")
    print(f"{'─'*70}")
    print(f"  {'#':>3}  {'Sim':>6}  {'Title':<48}  Tags")
    print(f"{'─'*70}")
    for i,r in enumerate(results,1):
        print(f"  {i:>3}  {r['sim']:>6.4f}  {r['title'][:47]:<48}  {r['tags'][:35]}")
    print(f"{'─'*70}")

In [ ]:
# ── Example: query by object ID ───────────────────────────────────────────
# Pick a random painting to query
sample_id = str(df.sample(1, random_state=SEED)['objectid'].values[0])
print(f'Querying object ID: {sample_id}')

for mode in ['semantic','structural','fused']:
    print(f'\n  Mode: {mode}')
    results = search(sample_id, mode=mode, top_k=5)
    display(sample_id, results)

In [ ]:
# ── Visualise results as a grid of images ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_grid(query_oid, results, mode='fused', n=5):
    qmeta = meta_dict.get(str(query_oid),{})
    fig, axes = plt.subplots(1, n+1, figsize=(3*(n+1), 3))
    fig.suptitle(f'Mode: {mode}', fontsize=11)
    # Query
    qpath = qmeta.get('local_path','')
    if qpath and Path(qpath).exists():
        axes[0].imshow(mpimg.imread(qpath))
    axes[0].set_title(f'QUERY\n{qmeta.get("title","")[:20]}', fontsize=7)
    axes[0].axis('off')
    # Results
    for i, r in enumerate(results[:n], 1):
        p = r.get('path','')
        if p and Path(p).exists(): axes[i].imshow(mpimg.imread(p))
        else: axes[i].set_facecolor('#eee')
        sim_str = f"{r['sim']:.3f}"
        axes[i].set_title(f'{sim_str}\n{r["title"][:20]}', fontsize=7)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

show_grid(sample_id, search(sample_id,'fused',5), mode='fused')
show_grid(sample_id, search(sample_id,'semantic',5), mode='semantic')
show_grid(sample_id, search(sample_id,'structural',5), mode='structural')

---
## Step 11 — Evaluation Metrics
Computes MAP@K, nDCG@K, Recall@K, Precision@1, and MRR for all three modes.
Ground truth: two paintings are relevant if they share at least one subject tag.

⏱ **~5 min**

In [ ]:
# ── Build ground truth from subject tags ──────────────────────────────────
print('Building ground-truth relevance sets...')
tag_col = 'subjectterms' if 'subjectterms' in df.columns else 'classification'
df['objectid'] = df['objectid'].astype(str)

tag_to_ids = {}
for _, row in tqdm(df.iterrows(), total=len(df), desc='Tag index'):
    oid = str(row['objectid'])
    for t in str(row.get(tag_col,'')).lower().split('|'):
        t = t.strip()
        if t: tag_to_ids.setdefault(t,set()).add(oid)

ground_truth = {}
for _, row in tqdm(df.iterrows(), total=len(df), desc='Relevance sets'):
    oid = str(row['objectid'])
    rel = set()
    for t in str(row.get(tag_col,'')).lower().split('|'):
        t = t.strip()
        if t and t in tag_to_ids: rel |= tag_to_ids[t]
    rel.discard(oid)
    if rel: ground_truth[oid] = rel

print(f'Paintings with relevant matches: {len(ground_truth):,}')
print(f'Avg relevant per query: {np.mean([len(v) for v in ground_truth.values()]):.1f}')

In [ ]:
# ── Metric functions ──────────────────────────────────────────────────────
def ap_at_k(retrieved, relevant, k):
    if not relevant: return 0.
    ap, nr = 0., 0
    for i,r in enumerate(retrieved[:k],1):
        if r in relevant: nr+=1; ap+=nr/i
    return ap/min(len(relevant),k)

def recall_at_k(retrieved, relevant, k):
    if not relevant: return 0.
    return sum(1 for r in retrieved[:k] if r in relevant)/len(relevant)

def ndcg_at_k(retrieved, relevant, k):
    if not relevant: return 0.
    dcg  = sum(1./np.log2(i+1) for i,r in enumerate(retrieved[:k],1) if r in relevant)
    idcg = sum(1./np.log2(i+1) for i in range(1,min(len(relevant),k)+1))
    return dcg/idcg if idcg>0 else 0.

def mrr(retrieved, relevant):
    for i,r in enumerate(retrieved,1):
        if r in relevant: return 1./i
    return 0.

In [ ]:
# ── Run evaluation for each mode ──────────────────────────────────────────
import json

valid_qids = [oid for oid in object_ids if oid in ground_truth]
n_queries  = min(EVAL_N_QUERIES, len(valid_qids))
query_ids  = random.sample(valid_qids, n_queries)
max_k      = max(EVAL_K_VALUES)

all_results = []
for mode in ['semantic','structural','fused']:
    embs    = mode_embeddings[mode]
    flat_ix = faiss.IndexFlatIP(embs.shape[1])
    flat_ix.add(embs)
    ap_s={k:[] for k in EVAL_K_VALUES}
    rc_s={k:[] for k in EVAL_K_VALUES}
    nd_s={k:[] for k in EVAL_K_VALUES}
    mrr_s=[]
    for qid in tqdm(query_ids, desc=f'Eval [{mode}]'):
        qrow = id_to_row.get(qid)
        if qrow is None: continue
        q_emb= embs[qrow:qrow+1].astype('float32')
        D,I  = flat_ix.search(q_emb, max_k+5)
        retr = []
        for ridx in I[0]:
            if ridx<0: continue
            rid = object_ids[int(ridx)]
            if rid!=qid: retr.append(rid)
            if len(retr)>=max_k: break
        rel = ground_truth[qid]
        mrr_s.append(mrr(retr,rel))
        for k in EVAL_K_VALUES:
            ap_s[k].append(ap_at_k(retr,rel,k))
            rc_s[k].append(recall_at_k(retr,rel,k))
            nd_s[k].append(ndcg_at_k(retr,rel,k))
    res = {'mode':mode, 'n_queries':n_queries, 'MRR':float(np.mean(mrr_s))}
    for k in EVAL_K_VALUES:
        res[f'MAP@{k}']    = float(np.mean(ap_s[k]))
        res[f'Recall@{k}'] = float(np.mean(rc_s[k]))
        res[f'nDCG@{k}']   = float(np.mean(nd_s[k]))
    all_results.append(res)

# Save
results_path = f'{RESULTS_DIR}/evaluation_results.json'
with open(results_path,'w') as f: json.dump(all_results,f,indent=2)
print(f'Results saved → {results_path}')

In [ ]:
# ── Print results table ────────────────────────────────────────────────────
cols = ['Mode','MRR'] + [f'MAP@{k}' for k in EVAL_K_VALUES] + [f'nDCG@{k}' for k in EVAL_K_VALUES] + [f'Recall@{k}' for k in EVAL_K_VALUES]
W = 10
print('  '+''.join(f'{c:<{W}}' for c in cols))
print('  '+'─'*(W*len(cols)))
for r in all_results:
    row = [r['mode'], f"{r['MRR']:.4f}"]
    for k in EVAL_K_VALUES: row.append(f"{r.get(f'MAP@{k}',0):.4f}")
    for k in EVAL_K_VALUES: row.append(f"{r.get(f'nDCG@{k}',0):.4f}")
    for k in EVAL_K_VALUES: row.append(f"{r.get(f'Recall@{k}',0):.4f}")
    print('  '+''.join(f'{v:<{W}}' for v in row))

best = max(all_results, key=lambda r: r.get(f'MAP@{EVAL_K_VALUES[-1]}',0))
print(f"\n  Best mode: '{best['mode']}'  MAP@{EVAL_K_VALUES[-1]} = {best.get(f'MAP@{EVAL_K_VALUES[-1]}',0):.4f}")

In [ ]:
# ── Plot metric comparison ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,3, figsize=(15,4))
metrics = ['MAP', 'nDCG', 'Recall']
colors  = ['#534AB7','#1D9E75','#D85A30']

for ax, metric in zip(axes, metrics):
    for res, col in zip(all_results, colors):
        vals = [res.get(f'{metric}@{k}',0) for k in EVAL_K_VALUES]
        ax.plot(EVAL_K_VALUES, vals, marker='o', label=res['mode'], color=col, linewidth=2)
    ax.set_title(metric, fontsize=13, fontweight='500')
    ax.set_xlabel('K', fontsize=11)
    ax.set_xticks(EVAL_K_VALUES)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Retrieval metric comparison by search mode', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Plot saved')